In [ ]:
import json
import os
import tempfile
from sklearn.metrics import accuracy_score, classification_report
import random
import requests
import statistics
import re
import nltk
from nltk import word_tokenize, pos_tag

In [54]:
#system prompt

generic_prompt = f"""You are an expert academic reviewer that provides actionable feedback on the paper contents provided in the user prompt. Present your feedback as a list of bullet points.
"""

#, such as "Clarify X", "Provide evidence for Y", or "Reorganize Z" for example.
specific_prompt = f"""You are an expert academic reviewer that provides actionable feedback on the paper contents provided in the user prompt. This means your feedback can be immediately implemented by the author to improve the paper. Frame each point as a specific, concrete suggestion or revision command."""

In [41]:
def build_prompt(paper):
  metadata = paper.get('metadata') #metadata dictionary that contains the actual contents of the paper
  content_list = metadata.get('sections')
  #print(content_list)
  #print(content_list[4].get('text'))
  #print(type(content_list[4]))

  paper_content = str(metadata.get('sections'))
  prompt = f"""Paper content: {paper_content}
  Provide your feedback on the paper. """

  return prompt


In [38]:
def model_forecasting(model, system_prompt, prompt):
    #print(prompt)
    # Send request to Ollama

    res = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model, #llama3.2:3b , "qwen3:latest"
            "system": system_prompt,
            "prompt": prompt, 
            "stream": False
            }
    )
    result = res.json()
    return result

In [52]:
def give_feedback(pdf_path, system_prompt, results):
    with open(pdf_path, 'r') as f1:
        paper = json.load(f1) #json file contents for one research paper

    prompt = build_prompt(paper)
    model = "llama3.2:latest"
    output = model_forecasting(model, system_prompt, prompt)
    json_response = output["response"]
    print(json_response)
    results[paper.get("name")] = {
        "review": json_response
    }
    return results

In [55]:
pdf_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\parsed_pdfs\\699.pdf.json"
review_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\acl_2017\\train\\reviews\\699.json"
results = {}
results = give_feedback(pdf_path, specific_prompt, results)
print(results)

Based on the provided text, I'll offer my feedback on the paper.

**Strengths:**

1. **Innovative approach**: The paper proposes an RNN-based generative model for predicting keyphrases in scientific texts, which is a novel application of the encoder-decoder framework.
2. **Comprehensive empirical studies**: The authors provide extensive experiments to evaluate the performance of their proposed model on various benchmark datasets, including scientific publications and news articles.
3. **Effective use of copy mechanism**: The incorporation of a copy mechanism allows the model to handle rarely-occurred phrases, which is a significant improvement over traditional keyphrase prediction models.

**Weaknesses:**

1. **Limited domain adaptation**: The model's performance on news articles is not as robust as its performance on scientific publications, indicating a need for further exploration of domain adaptation techniques.
2. **Lack of human evaluation**: While the authors mention human annot

In [ ]:
def generic_sample(dir_path, sample_size, output_path):
    paper_names = os.listdir(dir_path)
    results = {}
    for i in range(0, sample_size): 
        pdf_path = os.path.join(dir_path, paper_names[i])
        results = give_feedback(pdf_path, generic_prompt, results)
    
    with open(output_path,'a') as f3:
        json.dump(results,f3)


In [ ]:
def specific_sample(dir_path, sample_size, output_path):
    paper_names = os.listdir(dir_path)
    results = {}
    for i in range(0, sample_size): 
        pdf_path = os.path.join(dir_path, paper_names[i])
        results = give_feedback(pdf_path, specific_prompt, results)
    
    with open(output_path,'a') as f3:
        json.dump(results,f3)

In [48]:
dir_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\data\\iclr_2017\\train\\parsed_pdfs"
generic_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\generic_iclr_5"
specific_output_path = "C:\\Users\\G34371231\\OneDrive - The George Washington University\\Desktop\\PeerRead\\dtais_summer\\specific_iclr_5"
sample_size = 5

In [ ]:
generic_sample(dir_path, sample_size, generic_output_path)
specific_sample(dir_path, sample_size, specific_output_path)

# Evaluation

In [ ]:
# check for verbs
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

with open("output.json", "r") as f:
    results = json.load(f)

def extract_suggestions(text):
    """Extracts the Suggestions section and splits it into individual points."""
    suggestions_match = re.search(r"\*\*Suggestions:\*\*(.*?)\n\n", text, re.DOTALL)
    if not suggestions_match:
        return []

    suggestions_text = suggestions_match.group(1).strip()

    # Match numbered points: 1. ..., 2. ..., etc.
    points = re.findall(r"\d+\.\s+(.*?)(?=\n\d+\.|\Z)", suggestions_text, re.DOTALL)
    return [point.strip() for point in points if point.strip()]

def contains_verb(text):
    """Returns True if the text contains at least one verb."""
    tokens = word_tokenize(text)
    tags = pos_tag(tokens)
    return any(tag.startswith("VB") for _, tag in tags)  # VB, VBD, VBG, etc.

# Store actionable scores
actionable_scores = {}

for paper_name, content in results.items():
    review = content['review']
    points = extract_suggestions(review)
    total_points = len(points)
    points_with_verb = sum(contains_verb(point) for point in points)
    
    if total_points == 0:
        score = 0.0
    else:
        score = points_with_verb / total_points
    
    actionable_scores[paper_name] = {
        "total_points": total_points,
        "points_with_verb": points_with_verb,
        "actionable_score": round(score, 2)
    }

# Print or save results
for paper, score_data in actionable_scores.items():
    print(f"{paper}: {score_data}")
#check for verb word list

In [ ]:
#check for 